In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampNTZType, TimestampType, DoubleType, LongType

import pandas as pd

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
!powershell -Command "Invoke-WebRequest -Uri 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet' -OutFile 'yellow_tripdata_2025-11.parquet'"

Question 1: Install Spark and PySpark
Install Spark
Run PySpark
Create a local spark session
Execute spark.version.
What's the output?

In [4]:
print(spark.version)

3.5.1


Question 2: Yellow November 2025
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [ ]:
yellow_nov = spark.read.parquet('yellow_tripdata_2025-11.parquet')
yellow_nov = yellow_nov.repartition(4)
yellow_nov.write.parquet("data/raw/yellow/2025/11", mode="overwrite")
# 25MB


Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [8]:
yellow_nov.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [11]:
yellow_nov.createOrReplaceTempView("yellow_nov")
yellow_nov_report = spark.sql("""
SELECT COUNT(*) FROM yellow_nov WHERE date_trunc('day', tpep_pickup_datetime) = '2025-11-15'
""")
yellow_nov_report.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



Question 4: Longest trip
What is the length of the longest trip in the dataset in hours?

In [23]:
yellow_nov_report = spark.sql("""
SELECT datediff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime)/60 as duration, * FROM yellow_nov
                              WHERE tpep_dropoff_datetime > tpep_pickup_datetime
                              ORDER BY 1 DESC
""")
yellow_nov_report.show()

+------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|          duration|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
| 90.63333333333334|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00| 

Question 5: User Interface
Spark's User Interface which shows the application's dashboard runs on which local port?

4040

Question 6: Least frequent pickup location zone

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

Governor's Island/Ellis Island/Liberty Island
Arden Heights
Rikers Island
Jamaica Bay
If multiple answers are correct, select any

In [25]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')
df_zones.createOrReplaceTempView("zones")
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [29]:
least_popular_zones = spark.sql("""
SELECT y.PULocationID, z.Zone, COUNT(*) as count
FROM yellow_nov y JOIN zones z ON y.PULocationID = z.LocationID
WHERE y.passenger_count > 0
GROUP BY y.PULocationID, z.Zone
ORDER BY count ASC 
""")
least_popular_zones.show()

+------------+--------------------+-----+
|PULocationID|                Zone|count|
+------------+--------------------+-----+
|         105|Governor's Island...|    1|
|         172|New Dorp/Midland ...|    2|
|         251|         Westerleigh|    2|
|         204|   Rossville/Woodrow|    3|
|         176|             Oakwood|    3|
|           2|         Jamaica Bay|    4|
|         199|       Rikers Island|    4|
|         253|       Willets Point|    4|
|         206|Saint George/New ...|    4|
|         118|Heartland Village...|    5|
|          59|        Crotona Park|    5|
|          30|       Broad Channel|    5|
|         221|           Stapleton|    5|
|         156|     Mariners Harbor|    6|
|         214|South Beach/Donga...|    6|
|          27|Breezy Point/Fort...|    7|
|         115| Grymes Hill/Clifton|    8|
|         184|     Pelham Bay Park|   10|
|         120|     Highbridge Park|   17|
|          57|              Corona|   17|
+------------+--------------------